In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt


%pip install kagglehub catboost xgboost tqdm -q


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# # Task 1: Write your code here:
import numpy as np


df_clean = df.copy()

feature_cols = []

def delete_missings(df):
  for i in df:
    df[str(i)] = df[str(i)].fillna(df[str(i)].mean())
    if str(i) != "Target":
      feature_cols.append(i)

  # for i in df:
  #   if sum(df[str(i)]) == 0:
  #     df.drop(columns=[str(i)])

delete_missings(df_clean)
print(df_clean.isnull().sum())
print(type(feature_cols))


# Cleaned!

# I could have also deleted columns that have zeros, but I forgot and I can't use google to know :)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 3: Write your code here:

from sklearn.preprocessing import LabelEncoder

categorical_cols = df_clean.select_dtypes(include=["object"]).columns

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df_clean[feature_cols]
y = df_clean['Target']


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X, y)

In [ ]:
# Task 5: Write your code here:

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Target")

# Not balanced (before)

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from catboost import CatBoostClassifier


n_splits = 5

kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

accuracy_scores = []
f1_scores = []

for fold_idx, (train_index, test_index) in enumerate(kfold.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)



accuracy_scores = np.array(accuracy_scores)
f1_scores = np.array(f1_scores)
print(f1_scores)
print(accuracy_scores)

avg_loss = np.mean(f1_scores, axis=0)

print(avg_loss)

baseline_pred = np.full_like(y, y.mean())

# Evaluate the baseline
baseline_mae = f1_score(y_test, y_pred)

print(f"Baseline F1 (using f1 score): {baseline_mae:.4f}")

In [ ]:
# Task 2,3,4,5: Write your code here:

# All Done Above

In [ ]:
# Task 1: Write your code here:

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(100, 60))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

# The Golden Feature is P2!!

In [ ]:
# Task Bonus: Write your code here: